# Sugar in a soft drink from its density — Module 5 practical

*Your name, date.* Replace this line.

**Aim.** Calibrate density against sugar content with sucrose standards, then find the sugar content of one or more drinks, with an uncertainty.

**Data.** `density_raw.csv` (simulated example, made by `make_data.py`) or your own measurements from lesson 01.5 in the same columns. Rows with an empty `w_percent` are unknowns.

**Rules for this notebook.** Raw data are only read, never edited. Every number in the conclusion is computed below. Before you hand it in: *Kernel → Restart Kernel and Run All Cells*, and check that it runs top to bottom without errors.

In [ ]:
# Step 0: imports and a record of what produced these results
import sys
from datetime import date
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from uncertainties import correlated_values, ufloat

DATA = Path("density_raw.csv")          # relative path: the notebook lives next to the data
RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)
env = (f"{date.today().isoformat()}; Python {sys.version.split()[0]}; "
       + ", ".join(f"{p} {version(p)}" for p in ("numpy", "scipy", "pandas", "uncertainties")))
print(env)

## Step 1 — read the raw data and check it

Read the CSV with pandas. Print the shape, the dtypes and the number of missing values. Expect 45 rows and 8 columns for the example file; `w_percent` is empty (NaN) only for the unknowns.

In [ ]:
# TODO: raw = pd.read_csv(DATA); print raw.shape, raw.dtypes and raw.isna().sum()
raw = None

## Step 2 — densities

Each row is one repeat: `fills` syringe fills of `fill_volume_mL` each, weighed in a cup. Density = (full − empty) / (fills × fill volume). Do it in one vectorised line, in a **new** DataFrame (do not overwrite `raw`).

In [ ]:
# TODO: df = raw.copy(); df["rho_g_per_mL"] = ...
df = None

## Step 3 — mean and uncertainty per sample

Group by `sample` and compute the mean density, the SD and the count. Five repeats give a shaky SD, and all samples were measured the same way, so use the **pooled** SD, `sqrt(mean of the variances)`, divided by `sqrt(n)` as the standard error (SEM) of every mean. Split the table into standards (`w_percent` known) and unknowns.

In [ ]:
# TODO: summary = df.groupby("sample", sort=False).agg(w_percent=("w_percent", "first"),
#                 rho=("rho_g_per_mL", "mean"), sd=("rho_g_per_mL", "std"), n=("rho_g_per_mL", "size"))
# TODO: summary["sem"] = pooled SD / sqrt(n); std = rows with w_percent known; unk = the others
summary = std = unk = None

## Step 4 — fit two calibration models, weighted

Fit a straight line and a quadratic to the standards with `sigma = SEM` and `absolute_sigma=True`. Compare the reduced chi-square of the two (lesson 05.3).

In [ ]:
def line(w, a, b):
    return a + b * w

def quad(w, a, b, c):
    return a + b * w + c * w**2

def weighted_fit(f, x, y, s, p0):
    popt, pcov = curve_fit(f, x, y, p0=p0, sigma=s, absolute_sigma=True)
    chi2_red = np.sum(((y - f(x, *popt)) / s) ** 2) / (x.size - len(popt))
    return popt, pcov, chi2_red

x, y, s = std["w_percent"].to_numpy(float), std["rho"].to_numpy(), std["sem"].to_numpy()
# TODO: p_lin, c_lin, chi_lin = weighted_fit(line, x, y, s, [1.0, 0.004])
# TODO: p_quad, c_quad, chi_quad = weighted_fit(quad, x, y, s, [1.0, 0.004, 0.0]); print both chi2_red
p_lin = c_lin = chi_lin = p_quad = c_quad = chi_quad = None

## Step 5 — residuals decide

Plot the data with both models, and below it the normalised residuals (residual / SEM) of both. A good model leaves random scatter between about −2 and +2. Write one sentence: which model do you choose, and why?

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.4, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ww = np.linspace(0, x.max(), 100)
ax1.errorbar(x, y, yerr=s, fmt="o", ms=4, label="standards (mean ± SEM)")
# TODO: draw line(ww, *p_lin) and quad(ww, *p_quad) on ax1, and on ax2 the two residual series
#       (y - model(x)) / s against x, with a grey line at 0; label axes with units; add legends
ax1.set_ylabel("density (g/mL)")
ax2.set_xlabel("sucrose (% by mass)")
ax2.set_ylabel("residual / SEM")
fig.tight_layout()
fig.savefig(RUNS / "lesson05_calibration.png", dpi=120)

*Model choice (one sentence):* replace this line.

## Step 6 — the unknowns, with uncertainty

Invert the chosen model for each unknown. For the quadratic $\rho = a + b w + c w^2$ the root that lies inside the calibration range is $w = \dfrac{-b + \sqrt{b^2 - 4c(a - \rho)}}{2c}$. Use `correlated_values` for the parameters and `ufloat(mean, SEM)` for the measured density, so the uncertainty comes out automatically.

In [ ]:
# TODO: qa, qb, qc = correlated_values(p_quad, c_quad)
# TODO: for each row of unk: rho_u = ufloat(row["rho"], row["sem"]); w = the formula above;
#       store sample, density, w.n and w.s in a list of dicts called results and print them
results = []

## Step 7 — save the results next to the record of the environment

In [ ]:
pd.DataFrame(results).to_csv(RUNS / "lesson05_results.csv", index=False, float_format="%.5f")
(RUNS / "lesson05_environment.txt").write_text(env + "\n", encoding="utf-8")
print("saved:", sorted(p.name for p in RUNS.glob("lesson05_*")))

## Conclusion

Three to five sentences: the sugar content of each drink with its uncertainty, the model you used and why (residuals, reduced chi-square), the biggest source of uncertainty, and one thing this home method cannot detect (for example: other dissolved substances also raise the density). Then restart the kernel and run all cells once more.